# Making read stat file

In [1]:
# The OUT.transcript_model_reads.tsv.gz from IsoQuant can be directly used as the read stat input for IsoRanker
#/mmfs1/gscratch/stergachislab/asedeno/data/isoquant/isoquant_out/OUT/OUT.transcript_model_reads.tsv.gz

# Making classification file

In [ ]:
import pandas as pd

def parse_attributes(attribute_string):
        """
        Parse the attributes field in a GTF line and return a dictionary of key-value pairs.
        Example input: 'gene_id "ENSG00000310526.1"; gene_name "WASH7P";'
        Returns: {'gene_id': 'ENSG00000310526.1', 'gene_name': 'WASH7P'}
        """
        attrs = {}
        for attr in attribute_string.strip().split(';'):
            if attr.strip():
                key, value = attr.strip().split(' ', 1)
                attrs[key] = value.strip('"')
        return attrs

def extract_isoform_gene_mapping(gtf_path):
    """
    Extracts a mapping of transcript isoforms to their associated gene IDs and gene names from a GTF file.
    
    Parameters:
    - gtf_path: path to the input GTF file
    
    Returns:
    - pandas DataFrame with columns: Isoform, associated_ensg, associated_gene
    """
    gene_id_to_name = {}    # Dictionary to store gene_id -> gene_name mapping
    isoform_records = []    # List to store rows of final output (transcript_id, gene_id)

    # Read the GTF file line by line
    with open(gtf_path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue  # Skip header or comment lines
            
            parts = line.strip().split('\t')
            feature_type = parts[2]  # e.g., 'gene', 'transcript', 'exon'
            attributes = parse_attributes(parts[8])  # Parse the 9th column

            if feature_type == 'gene':
                # Extract gene_id and gene_name to build a lookup
                gene_id = attributes.get('gene_id')
                gene_name = attributes.get('gene_name')
                if gene_id and gene_name:
                    gene_id_to_name[gene_id] = gene_name

            elif feature_type == 'transcript':
                # Extract transcript_id and gene_id
                transcript_id = attributes.get('transcript_id')
                gene_id = attributes.get('gene_id')
                if transcript_id and gene_id:
                    isoform_records.append({
                        'isoform': transcript_id,
                        'associated_ensg': gene_id
                    })

    # Convert list of transcript records to a DataFrame
    df = pd.DataFrame(isoform_records)

    # Add associated_gene column by mapping gene_id → gene_name using the lookup
    df['associated_gene'] = df['associated_ensg'].map(gene_id_to_name)

    return df


# Run the function to extract the data
classification_data = extract_isoform_gene_mapping("/mmfs1/gscratch/stergachislab/asedeno/data/isoquant/isoquant_out/OUT/OUT.transcript_models.gtf")

# Add columns to replicate pigeon classification output
classification_data["structural_category"] = ""
classification_data["subcategory"] = ""

classification_data.to_csv("IsoQuant_classification.txt", sep="\t", index=False)

# Running IsoRanker

In [ ]:
# Use the created read_stat and classification files as inputs for IsoRanker: https://github.com/yhhc2/IsoRanker/blob/main/examples/Expression_AllelicImbalance_NMD/Expression_AllelicImbalance_NMD.ipynb